# 6. RAG(Retrieval Augmented Generation): 검색 증강 생성
    - 검색하는 질문과 관련 있는 문서들을 함께 prompt로 보내는 방법.

## 방식
1. Stuff
2. Refine
3. Map Reduce

## 단계
1. [Retrieval](https://python.langchain.com/docs/integrations/document_loaders/#all-document-loaders)
![](https://python.langchain.com/v0.1/assets/images/data_connection-95ff2033a8faa5f3ba41376c0f6dd32a.jpg)

## 6.2 [Tiktoken](https://platform.openai.com/tokenizer)
- 모델이 텍스트를 세는 방법
- Python Standard Library function인 len이 글자수를 세는 방법과는 다르다.

In [ ]:
# 6.1 Data Loaders and Splitters

from langchain.document_loaders import TextLoader, PyPDFLoader, UnstructuredEPubLoader, UnstructuredFileLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

txt_loader = TextLoader("../files/Bread.txt")
pdf_loader = PyPDFLoader("../files/Bread.pdf")
loader = UnstructuredFileLoader("../files/Bread.epub")

splitter = RecursiveCharacterTextSplitter( 
    chunk_size=500,
    chunk_overlap=50, # 문장이나 문단을 분할할 때 앞 조각 일부분을 가져온다.
) 

splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=600,
    chunk_overlap=100,
    length_function = len, # This is the default value. The Python Standard Library function, len.
)

# 글 길이 셀 때 len function 대신 Tokenizer function 사용
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n\n",
    chunk_size=600,
    chunk_overlap=100,
)

# 1번째 방법
docs = loader.load()
splitter.split_documents(docs)

# 2번째 방법
splitted = loader.load_and_split(text_splitter=splitter)
len(splitted)

## 6.3 Vectors
### embedding
    - 사람이 읽는 텍스트를 컴퓨터가 인식할 수 있는 숫자로 변환하는 작업
    - 이전 단계에서 split한 문서 마다 embed 예정
### [vectorization](https://turbomaze.github.io/word2vecjson/)
    - 단어의 특성을 나타내는 1000개의 차원에, 각 단어가 해당 특성을 얼마나 반영하는지 숫자로 평가하여 좌표를 찍는다.
    - 추천 알고리즘은 비슷한 특성을 가진 가까운 벡터를 추천한다.

In [ ]:
from langchain.embeddings import OpenAIEmbeddings # Open AI Embed Model

embedder = OpenAIEmbeddings()
vector = embedder.embed_documents([])
len(vector[0]) # How many vectors the query have.

1536

## 6.4 Vector Store
    - 벡터공간은 일종의 데이터베이스.
    - 클라우드 환경 vs 로컬 컴퓨터 / 유료 vs 무료 옵션이 있다.
    - (ex) Chroma(로컬, 오픈소스), FAISS, pinecone(클라우드)
    1. 모든 문서를 벡터로 변환한다.
    2. 임베딩한 문서를 캐싱한다.
    3. 벡터공간에서 질문과 관련된 문서들을 검색할 수 있다.

### CacheBackedEmbeddings
    1. cache에 embeddings이 이미 존재하는지 확인한다.
    2. 존재하지 않는다면 vectorstore를 호출할 때 documents와 OpenAIEmbeddings를 사용하여 cache에 저장한다.
    3. 존재한다면 캐시에 저장된 embeddings를 가져온다.

In [ ]:
from langchain.chat_models.openai import ChatOpenAI
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.vectorstores.chroma import Chroma
from langchain.storage import LocalFileStore

cache_dir = LocalFileStore("./.cache/")

loader = UnstructuredFileLoader("../files/Bread.txt")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n\n",
    chunk_size=600,
    chunk_overlap=100,
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, cache_dir
)

docs = loader.load_and_split(text_splitter=splitter)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

vectorstore.similarity_search("The order of adding ingrediants") # 관련 문서를 찾아온다.

## 6.6 RetrievalQA
- off-the-shelf chain
- types:
1. StuffDocumentsChain
    - prompt에 찾은 documents를 모두 넣어 model에 제공한다.
2. RefineDocumentsChain
    - model이 각 document를 읽으며 답변을 생성한다. 답변을 쌓으면서 가다듬는다.
3. MapReduceDocumentsChain
    - 각 document를 요약하여 llm에 각각 전달한다.
4. Map re-rank documents chain
    - 각 document를 읽으면서 답변을 생성하고 그 답변에 대해 점수를 매긴다. 가장 높은 점수를 획득한 답변과 그 점수를 함께 반환한다.

In [ ]:
# 6.8 Stuff LCEL Chain

from langchain.chat_models.openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.document_loaders import UnstructuredFileLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.storage import LocalFileStore
from langchain.vectorstores.faiss import FAISS
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.schema.runnable import RunnablePassthrough

cache_dir = LocalFileStore("./.cache/")
llm = ChatOpenAI(
    temperature=0.1,
)

loader = UnstructuredFileLoader("../files/Bread.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n\n",
    chunk_size=600,
    chunk_overlap=100,
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

docs = loader.load_and_split(text_splitter=splitter)

vectorstore = FAISS.from_documents(docs, cached_embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    # 3
    ("system", "You are a helpful assistant. Answer questions using only the following context. If you don't know the answer just say you don't know, don't make it up:\n\n{context}"),
    # 1 -> 4 delievered by RunnabePassthrough
    ("human", "{question}"),
])

     # 2 -> output: list of documents
chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

# 1
chain.invoke("Explain the oven heat.")